In [ ]:
"""
CH4 comparison: AIRS3STD TotCH4_A  vs.  CAMS total_column_methane (Mace Head).

Pipeline:
  1. Load both CSVs, aggregate to monthly means (re-using the same procedure
     used in the AIRS3STD and CAMS_single trend pipelines).
  2. Convert AIRS TotCH4_A from molecules cm-2 to mg m-2:
            C [mg m-2] = N [molec cm-2] * 1e4 * (M_CH4 / N_A) * 1e3
            M_CH4 = 16.043   g mol-1
            N_A   = 6.02214076e23 molecules mol-1
     This gives a single multiplicative factor ≈ 2.6640e-16.
  3. Left-join (pandas merge how='left') AIRS and CAMS monthly series on date.
     The AIRS index is the "left" table.
     -> write D:/PLOTS/CH4_comparison/ch4CAMSvsAIRSTD.csv
  4. Per series (AIRS, CAMS):
        - 4-panel STL decomposition           (01_stl_decomposition.png)
        - QR tau-grid                          (02_tau_grid.png)
        - QR slopes at tau = 0.25/0.50/0.75    (03_qr_three_quantiles.png)
        - trend_summary.csv
  5. Combined plots:
        - z-score STL trends, both series      (05_zscore_trend_combined.png)
        - publication TIFF, two panels         (06_timeseries_scatter.tiff)
            A) two full monthly time series, x labels rotated 90°, grid
            B) scatter w/ 1:1 line, regression line, inset (upper-right):
               y = a*x + b, Pearson R
"""

from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from matplotlib.ticker import MaxNLocator
from scipy.stats import pearsonr
from statsmodels.regression.quantile_regression import QuantReg
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ============================================================================
# CONFIG ---------------------------------------------------------------------
# ============================================================================
AIRS_CSV  = Path(r"D:/Time Series/AIRS3STD.006_mace_head.csv")
CAMS_CSV  = Path(r"D:/Time Series/CAMS_single_mace_head_main.csv")
OUT_ROOT  = Path(r"D:/PLOTS/CH4_comparison")
MERGED_CSV = OUT_ROOT / "ch4CAMSvsAIRSTD.csv"

AIRS_VAR  = "TotCH4_A"
CAMS_VAR  = "total_column_methane"

# Unit-conversion constants (AIRS column -> mg m-2).
M_CH4 = 16.043           # g mol-1
N_A   = 6.02214076e23    # molecules mol-1
# C [mg m-2] = N [molec cm-2] * 1e4 (cm-2 -> m-2) * (M/NA) (molec -> g) * 1e3 (g -> mg)
AIRS_TO_MG_M2 = 1.0e4 * (M_CH4 / N_A) * 1.0e3   # ≈ 2.6640e-16

# Display colours (consistent z-score / TIFF).
COL_AIRS = "#4E342E"   # dark brown
COL_CAMS = "#0277BD"   # azure

# STL base case (Moschos et al.).
STL_PERIOD, STL_SEASONAL, STL_TREND, STL_ROBUST = 12, 7, 23, True

# Quantile grid.
TAU_GRID = np.round(np.arange(0.05, 0.96, 0.05), 2)
TAU_KEY  = (0.25, 0.50, 0.75)

# HAC lag for Wald CIs on QR slopes.
HAC_MAXLAGS = 12

# Imputation rule (same as upstream pipelines).
COVERAGE_THRESHOLD = 0.25

# Common axis label after conversion.
UNIT_LABEL = r"mg m$^{-2}$"

# Matplotlib defaults.
plt.rcParams.update({
    "figure.dpi":         110,
    "savefig.dpi":        160,
    "savefig.bbox":       "tight",
    "font.size":          9,
    "axes.titlesize":     10,
    "axes.labelsize":     9,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.color":         "#e6e6e6",
    "grid.linewidth":     0.6,
    "legend.frameon":     False,
    "legend.fontsize":    8,
})


# ============================================================================
# LOADERS --------------------------------------------------------------------
# ============================================================================

def _find_header_row(csv_path: Path) -> tuple[int, str]:
    """Locate the column-header line in an AIRS-style CSV with a metadata block."""
    candidates = []
    with open(csv_path, "r", encoding="utf-8-sig", errors="replace") as fh:
        for i, raw in enumerate(fh):
            line = raw.rstrip("\r\n")
            if not line.strip():
                continue
            stripped = line.lstrip("#").strip()
            if not stripped:
                continue
            sep = "\t" if "\t" in stripped else ("," if "," in stripped else None)
            first_tok = stripped.split(sep)[0].strip().lower() if sep else stripped.lower()
            if first_tok == "date":
                return i, sep or "\t"
            candidates.append((i, line[:80]))
            if i > 200:
                break
    raise RuntimeError(
        f"Could not locate a 'date' header row in {csv_path}. "
        f"First non-comment lines:\n  " +
        "\n  ".join(f"line {ln}: {txt!r}" for ln, txt in candidates[:5])
    )


def load_airs_monthly(csv_path: Path, var: str
                       ) -> tuple[pd.Series, pd.Series]:
    """Read AIRS daily CSV; return (monthly_raw, monthly_coverage) for `var`,
    already in mg m-2 (i.e. after applying AIRS_TO_MG_M2).
    """
    header_row, sep = _find_header_row(csv_path)
    df = pd.read_csv(csv_path, sep=sep, header=header_row,
                     encoding="utf-8-sig", engine="python",
                     na_values=["NA", "NaN", "nan", "-9999.0", "-9999"])
    df.columns = [str(c).lstrip("#").strip() for c in df.columns]
    if "date" not in df.columns:
        raise RuntimeError(f"AIRS CSV: 'date' column not found. Got: "
                           f"{list(df.columns)[:6]} ...")
    if var not in df.columns:
        raise RuntimeError(f"AIRS CSV: variable {var!r} not found. "
                           f"Sample columns: {list(df.columns)[:12]}")

    # Parse 'YYYY.MM.DD' (with fallbacks).
    raw_date = df["date"].astype(str).str.strip()
    parsed = pd.Series(pd.NaT, index=raw_date.index, dtype="datetime64[ns]")
    for fmt in ("%Y.%m.%d", "%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y"):
        mask = parsed.isna()
        if not mask.any():
            break
        attempt = pd.to_datetime(raw_date[mask], format=fmt, errors="coerce")
        parsed.loc[mask] = attempt
    mask = parsed.isna()
    if mask.any():
        parsed.loc[mask] = pd.to_datetime(raw_date[mask], errors="coerce")
    df["date"] = parsed
    df = df.dropna(subset=["date"]).sort_values("date").drop_duplicates("date")
    df = df.set_index("date")

    s = pd.to_numeric(df[var], errors="coerce") * AIRS_TO_MG_M2     # -> mg m-2

    monthly  = s.resample("ME").mean()
    counts   = s.resample("ME").count()
    expected = pd.Series(monthly.index.days_in_month, index=monthly.index)
    coverage = counts.div(expected).clip(upper=1.0)
    return monthly.rename(var), coverage.rename(var)


def load_cams_monthly(csv_path: Path, var: str
                       ) -> tuple[pd.Series, pd.Series, float]:
    """Read CAMS hourly CSV; return (monthly_raw, monthly_coverage, step_h)."""
    df = pd.read_csv(csv_path)
    if "datetime_utc" not in df.columns:
        raise RuntimeError(f"CAMS CSV: 'datetime_utc' column not found. Got: "
                           f"{list(df.columns)[:6]} ...")
    if var not in df.columns:
        raise RuntimeError(f"CAMS CSV: variable {var!r} not found.")

    dt = pd.to_datetime(df["datetime_utc"], errors="coerce", utc=True)
    df = df.assign(datetime_utc=dt.dt.tz_convert(None)).dropna(subset=["datetime_utc"])
    df = df.sort_values("datetime_utc").drop_duplicates("datetime_utc").set_index("datetime_utc")

    deltas = pd.Series(df.index).diff().dropna().dt.total_seconds() / 3600.0
    step_h = float(deltas.median()) if len(deltas) else 3.0
    if not np.isfinite(step_h) or step_h <= 0:
        step_h = 3.0

    s = pd.to_numeric(df[var], errors="coerce")             # kg m-2
    s = s * 1.0e6                                            # -> mg m-2

    monthly  = s.resample("ME").mean()
    counts   = s.resample("ME").count()
    expected = pd.Series((24.0 / step_h) * monthly.index.days_in_month,
                         index=monthly.index)
    coverage = counts.div(expected).clip(upper=1.0)
    return monthly.rename(var), coverage.rename(var), step_h


# ============================================================================
# IMPUTATION / STL / AR / QR -------------------------------------------------
# ============================================================================

def impute_calendar_median(series: pd.Series, coverage: pd.Series,
                            threshold: float) -> tuple[pd.Series, pd.Series]:
    ser  = series.copy()
    cov  = coverage.reindex(ser.index).fillna(0.0)
    bad  = ser.isna() | (cov < threshold)
    good = ser[~bad]
    months = ser.index.month
    if good.empty:
        ser[bad] = np.nan
    else:
        month_med = good.groupby(good.index.month).median()
        for i in np.where(bad.values)[0]:
            ser.iloc[i] = month_med.get(months[i], good.median())
    return ser, bad


def stl_decompose(series: pd.Series):
    return STL(series.values, period=STL_PERIOD, seasonal=STL_SEASONAL,
               trend=STL_TREND, robust=STL_ROBUST).fit()


def ar_correct_residuals(resid: np.ndarray) -> tuple[np.ndarray, int, float]:
    r = pd.Series(resid).dropna().values
    if len(r) < 30:
        return resid, 0, np.nan
    try:
        lb = acorr_ljungbox(r, lags=[min(24, len(r) // 4)], return_df=True)
        p_value = float(lb["lb_pvalue"].iloc[0])
    except Exception:
        return resid, 0, np.nan
    if not np.isfinite(p_value) or p_value >= 0.05:
        return resid, 0, p_value
    best = None
    for p in range(1, 11):
        try:
            fit = AutoReg(r, lags=p, old_names=False).fit()
            if best is None or fit.aic < best[1]:
                best = (p, fit.aic, fit)
        except Exception:
            continue
    if best is None:
        return resid, 0, p_value
    p_used, _aic, fit = best
    inn = np.asarray(fit.resid)
    full = np.full_like(r, np.nan, dtype=float)
    full[p_used:] = inn
    out = np.full(len(resid), np.nan, dtype=float)
    src_i = 0
    for i, val in enumerate(resid):
        if np.isnan(val):
            continue
        out[i] = full[src_i]
        src_i += 1
    return out, p_used, p_value


def qr_slope_with_hac(t_years: np.ndarray, y: np.ndarray, tau: float,
                       maxlags: int = HAC_MAXLAGS) -> tuple[float, float, float]:
    X = np.column_stack([np.ones_like(t_years), t_years])
    mod = QuantReg(y, X)
    try:
        res = mod.fit(q=tau, vcov="kernel", kernel="hac",
                      bandwidth=maxlags, max_iter=5000)
    except Exception:
        res = mod.fit(q=tau, max_iter=5000)
    b1   = float(res.params[1])
    se   = float(res.bse[1])
    half = 1.96 * se
    return b1, b1 - half, b1 + half


# ============================================================================
# PER-SERIES PLOTS (one folder per series) -----------------------------------
# ============================================================================

def plot_stl_decomposition(pretty: str, color: str, unit_label: str,
                            monthly_raw: pd.Series, monthly_imp: pd.Series,
                            mask: pd.Series, stl_res, resid_corr: np.ndarray,
                            ar_p: int, lb_p: float, out_path: Path) -> None:
    idx   = monthly_imp.index
    raw_v = monthly_raw.values.astype(float)
    imp_v = monthly_imp.values.astype(float)

    fig, axes = plt.subplots(4, 1, figsize=(9.0, 9.6), sharex=True)
    fig.suptitle(f"{pretty}  —  STL decomposition (base case)",
                 y=0.995, fontsize=11, fontweight="bold", color=color)

    ax = axes[0]
    ax.plot(idx, imp_v, ls="--", color="#999999", lw=1.0,
            label="Post-imputation continuous")
    raw_only = np.where(mask.values, np.nan, raw_v)
    ax.plot(idx, raw_only, color="black", lw=1.1, label="Raw monthly")
    ax.set_title("(a) Raw vs. Post-Imputation Monthly Data", loc="left")
    ax.set_ylabel(unit_label)
    ax.legend(loc="upper left", ncol=2)

    ax = axes[1]
    ax.plot(idx, stl_res.trend, color="#1f6f9c", lw=1.6, label="Trend")
    ax.set_title("(b) STL Trend Component", loc="left")
    ax.set_ylabel(unit_label)
    ax.legend(loc="upper left")

    ax = axes[2]
    ax.plot(idx, stl_res.seasonal, color="#e1a126", lw=1.0, label="Seasonal")
    ax.set_title("(c) STL Seasonal Component", loc="left")
    ax.set_ylabel(unit_label)
    ax.legend(loc="upper left")

    ax = axes[3]
    ax.plot(idx, stl_res.resid, ls="--", color="#7a7a7a", lw=0.9,
            label="Raw Residuals")
    if ar_p > 0 and np.any(np.isfinite(resid_corr)):
        ax.plot(idx, resid_corr, color="#2a9d8f", lw=1.0,
                label=f"AR({ar_p})-Corrected Residuals")
    else:
        ax.plot(idx, stl_res.resid, color="#2a9d8f", lw=1.0,
                label="Residuals (no AC, no correction)")
    ax.axhline(0, color="#bbbbbb", lw=0.6)
    title = "(d) Residuals"
    if np.isfinite(lb_p):
        title += f"   [Ljung–Box p = {lb_p:.3g}]"
    ax.set_title(title, loc="left")
    ax.set_ylabel(unit_label)
    ax.set_xlabel("Year")
    ax.legend(loc="upper left", ncol=2)

    fig.tight_layout(rect=[0, 0, 1, 0.99])
    fig.savefig(out_path)
    plt.close(fig)


def plot_tau_grid(pretty: str, color: str, unit_label: str,
                   t_years: np.ndarray, trend_vals: np.ndarray,
                   tau_grid: np.ndarray, ref_value: float,
                   out_path: Path) -> pd.DataFrame:
    rows = []
    for tau in tau_grid:
        b, lo, hi = qr_slope_with_hac(t_years, trend_vals, float(tau))
        rows.append({"tau": tau, "slope": b, "ci_low": lo, "ci_high": hi})
    df_tau = pd.DataFrame(rows)

    rel = 100.0 / ref_value if (np.isfinite(ref_value) and ref_value > 0) else np.nan

    fig, axes = plt.subplots(2, 1, figsize=(7.5, 7.0), sharex=True)
    fig.suptitle(f"{pretty}  —  STL-Trend QR  τ-grid",
                 y=0.995, fontsize=11, fontweight="bold", color=color)

    ax = axes[0]
    ax.fill_between(df_tau["tau"], df_tau["ci_low"], df_tau["ci_high"],
                    alpha=0.25, color=color, label="95% CI")
    ax.plot(df_tau["tau"], df_tau["slope"], "o-", color=color, lw=1.4,
            ms=4.5, label="QR Slope")
    ax.axhline(0, ls="--", color="#888888", lw=0.7)
    ax.set_ylabel(f"STL-Trend QR Slope ({unit_label}/year)")
    ax.legend(loc="best")

    ax = axes[1]
    if np.isfinite(rel):
        ax.fill_between(df_tau["tau"],
                        df_tau["ci_low"]  * rel,
                        df_tau["ci_high"] * rel,
                        alpha=0.25, color=color, label="95% CI")
        ax.plot(df_tau["tau"], df_tau["slope"] * rel, "o-",
                color=color, lw=1.4, ms=4.5, label="QR Percent Change")
        ax.axhline(0, ls="--", color="#888888", lw=0.7)
        ax.set_ylabel("Percent Change per Year (%)")
    else:
        ax.text(0.5, 0.5,
                "Reference value not available\n"
                "(zero, signed, or non-finite mean) — relative slope not shown.",
                transform=ax.transAxes, ha="center", va="center", color="#888")
        ax.set_yticks([])
    ax.set_xlabel("Quantile (τ)")
    ax.legend(loc="best")

    fig.tight_layout(rect=[0, 0, 1, 0.99])
    fig.savefig(out_path)
    plt.close(fig)
    return df_tau


def plot_three_quantiles(pretty: str, color: str, unit_label: str,
                          df_tau: pd.DataFrame, out_path: Path) -> pd.DataFrame:
    sub = df_tau[df_tau["tau"].round(2).isin([round(t, 2) for t in TAU_KEY])].copy()
    sub = sub.sort_values("tau").reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(5.6, 4.0))
    fig.suptitle(f"{pretty}  —  QR slopes at τ = 0.25, 0.50, 0.75",
                 y=0.995, fontsize=11, fontweight="bold", color=color)

    xs = np.arange(len(sub))
    err_low  = (sub["slope"] - sub["ci_low"]).values
    err_high = (sub["ci_high"] - sub["slope"]).values

    ax.errorbar(xs, sub["slope"].values, yerr=[err_low, err_high],
                fmt="o", color=color, ecolor=color,
                capsize=4, lw=1.5, ms=7,
                label="QR slope ± 95% CI (HAC)")
    ax.axhline(0, ls="--", color="#888888", lw=0.7)
    ax.set_xticks(xs)
    ax.set_xticklabels([f"τ = {t:.2f}" for t in sub["tau"]])
    ax.set_ylabel(f"STL-Trend QR Slope ({unit_label}/year)")
    ax.legend(loc="best")
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6))

    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(out_path)
    plt.close(fig)
    return sub


# ============================================================================
# COMBINED PLOTS -------------------------------------------------------------
# ============================================================================

def plot_zscore_combined(idx_a: pd.DatetimeIndex, z_a: np.ndarray, lbl_a: str,
                          idx_c: pd.DatetimeIndex, z_c: np.ndarray, lbl_c: str,
                          out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(9.0, 3.6))
    ax.plot(idx_a, z_a, color=COL_AIRS, lw=1.8, label=lbl_a)
    ax.plot(idx_c, z_c, color=COL_CAMS, lw=1.8, label=lbl_c)
    ax.axhline(0, ls="--", color="#888", lw=0.6)
    ax.set_title("CH$_4$ Total Column  —  Standardised STL Trend (z-score)",
                 fontweight="bold", loc="left")
    ax.set_ylabel("STL Trend (z-score)")
    ax.set_xlabel("Year")
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(out_path)
    plt.close(fig)


def plot_publication_tiff(merged: pd.DataFrame, out_path: Path) -> None:
    """Nature-ready two-panel TIFF: (A) full time series, (B) scatter."""
    fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8),
                             gridspec_kw={"width_ratios": [1.55, 1.0]})

    # ---- Panel A: time series -------------------------------------------------
    ax = axes[0]
    ax.plot(merged.index, merged["AIRS_TotCH4_A_mg_m2"],
            color=COL_AIRS, lw=1.4, label="AIRS3STD TotCH$_4$ (asc.)")
    ax.plot(merged.index, merged["CAMS_total_column_methane_mg_m2"],
            color=COL_CAMS, lw=1.4, label="CAMS total column CH$_4$")
    ax.set_ylabel(f"CH$_4$ total column ({UNIT_LABEL})")
    ax.set_xlabel("Date")
    ax.set_title("(A) Monthly time series", loc="left", fontweight="bold")
    ax.grid(True, color="#e6e6e6", lw=0.6)
    ax.legend(loc="best")

    # Yearly major ticks, all rotated 90°.
    ax.xaxis.set_major_locator(mdates.YearLocator(1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.setp(ax.get_xticklabels(), rotation=90, ha="center")

    # ---- Panel B: scatter ----------------------------------------------------
    ax = axes[1]
    pair = merged[["AIRS_TotCH4_A_mg_m2", "CAMS_total_column_methane_mg_m2"]].dropna()
    x = pair["AIRS_TotCH4_A_mg_m2"].values
    y = pair["CAMS_total_column_methane_mg_m2"].values

    ax.scatter(x, y, s=14, color="#444444", alpha=0.65,
                edgecolor="none", label=f"Monthly pairs (n = {len(pair)})")

    # Common range for 1:1 line.
    lo = float(np.nanmin([x.min(), y.min()]))
    hi = float(np.nanmax([x.max(), y.max()]))
    pad = 0.02 * (hi - lo) if hi > lo else 1.0
    rng = np.array([lo - pad, hi + pad])
    ax.plot(rng, rng, ls="--", color="#888888", lw=1.0, label="1 : 1")

    # OLS regression line (y on x) + Pearson R.
    if len(pair) >= 2 and np.std(x) > 0:
        a, b = np.polyfit(x, y, 1)
        r, _ = pearsonr(x, y)
        xs = np.linspace(rng[0], rng[1], 200)
        ax.plot(xs, a * xs + b, color="#C62828", lw=1.4, label="OLS fit")
        sign = "+" if b >= 0 else "−"
        text = (f"y = {a:.2f}·x {sign} {abs(b):.2f}\n"
                f"Pearson R = {r:.2f}")
    else:
        text = "Insufficient overlap for fit"

    # Inset box in upper-right corner of axes.
    ax.text(0.97, 0.97, text, transform=ax.transAxes,
            ha="right", va="top",
            fontsize=9,
            bbox=dict(boxstyle="round,pad=0.4",
                      facecolor="white", edgecolor="#999999", lw=0.8))

    ax.set_xlim(rng); ax.set_ylim(rng)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(f"AIRS3STD TotCH$_4$ ({UNIT_LABEL})")
    ax.set_ylabel(f"CAMS CH$_4$ ({UNIT_LABEL})")
    ax.set_title("(B) AIRS vs. CAMS — monthly means", loc="left", fontweight="bold")
    ax.grid(True, color="#e6e6e6", lw=0.6)
    ax.legend(loc="lower right")

    fig.tight_layout()
    # 300 dpi TIFF with LZW compression for publication.
    fig.savefig(out_path, dpi=300, format="tiff",
                pil_kwargs={"compression": "tiff_lzw"})
    plt.close(fig)


# ============================================================================
# PER-SERIES DRIVER ----------------------------------------------------------
# ============================================================================

def run_single_series(label: str, pretty: str, color: str,
                       monthly_raw: pd.Series, coverage: pd.Series,
                       out_dir: Path) -> dict:
    """Full per-series pipeline. Returns dict of trend stats + STL trend."""
    out_dir.mkdir(parents=True, exist_ok=True)

    series_imp, mask_var = impute_calendar_median(monthly_raw, coverage,
                                                    COVERAGE_THRESHOLD)
    n_finite = int(np.isfinite(series_imp.values).sum())
    if n_finite < 2 * STL_PERIOD:
        raise RuntimeError(f"{label}: too few finite months ({n_finite})")
    if series_imp.isna().any():
        residual_mask = series_imp.isna()
        series_imp = series_imp.ffill().bfill()
        mask_var   = (mask_var | residual_mask)

    stl_res = stl_decompose(series_imp)
    resid_corr, ar_p, lb_p = ar_correct_residuals(np.asarray(stl_res.resid))

    idx     = series_imp.index
    t_years = (idx - idx[0]).days / 365.25
    trend_v = np.asarray(stl_res.trend)
    ref_val = float(np.nanmean(series_imp.values))

    plot_stl_decomposition(pretty, color, UNIT_LABEL,
                            monthly_raw, series_imp, mask_var,
                            stl_res, resid_corr, ar_p, lb_p,
                            out_dir / "01_stl_decomposition.png")
    df_tau   = plot_tau_grid(pretty, color, UNIT_LABEL, t_years, trend_v,
                              TAU_GRID, ref_val,
                              out_dir / "02_tau_grid.png")
    df_three = plot_three_quantiles(pretty, color, UNIT_LABEL, df_tau,
                                      out_dir / "03_qr_three_quantiles.png")

    # trend_summary.csv (full tau grid + key tau slopes + percent/yr).
    summary = df_tau.copy()
    summary.insert(0, "series", label)
    summary.insert(1, "unit", UNIT_LABEL)
    rel = 100.0 / ref_val if (np.isfinite(ref_val) and ref_val > 0) else np.nan
    summary["pct_per_year"] = summary["slope"]  * rel
    summary["pct_ci_low"]   = summary["ci_low"] * rel
    summary["pct_ci_high"]  = summary["ci_high"]* rel
    summary["ref_value"]    = ref_val
    summary["ar_p"]         = ar_p
    summary["ljungbox_p"]   = lb_p
    summary.to_csv(out_dir / "trend_summary.csv", index=False)

    # z-score of STL trend for the combined plot.
    mu, sd = np.nanmean(trend_v), np.nanstd(trend_v)
    z = (trend_v - mu) / sd if (sd and np.isfinite(sd) and sd > 0) else np.full_like(trend_v, np.nan)

    return {
        "label":     label,
        "ref_value": ref_val,
        "n_months":  int(series_imp.shape[0]),
        "n_imputed": int(mask_var.sum()),
        "ar_p":      ar_p,
        "ljungbox_p": lb_p,
        "stl_trend_index":  idx,
        "stl_trend_values": trend_v,
        "stl_trend_z":      z,
        "df_three":  df_three,
    }


# ============================================================================
# MAIN -----------------------------------------------------------------------
# ============================================================================

def main():
    if not AIRS_CSV.exists(): raise SystemExit(f"Not found: {AIRS_CSV}")
    if not CAMS_CSV.exists(): raise SystemExit(f"Not found: {CAMS_CSV}")
    OUT_ROOT.mkdir(parents=True, exist_ok=True)

    print(f"Reading AIRS: {AIRS_CSV}")
    airs_monthly, airs_cov = load_airs_monthly(AIRS_CSV, AIRS_VAR)
    print(f"  AIRS months: {len(airs_monthly)} "
          f"({airs_monthly.index.min().strftime('%Y-%m')} -> "
          f"{airs_monthly.index.max().strftime('%Y-%m')})")
    print(f"  Conversion factor (molec cm-2 -> mg m-2) = {AIRS_TO_MG_M2:.6e}")

    print(f"\nReading CAMS: {CAMS_CSV}")
    cams_monthly, cams_cov, step_h = load_cams_monthly(CAMS_CSV, CAMS_VAR)
    print(f"  CAMS native step ≈ {step_h:.2f} h")
    print(f"  CAMS months: {len(cams_monthly)} "
          f"({cams_monthly.index.min().strftime('%Y-%m')} -> "
          f"{cams_monthly.index.max().strftime('%Y-%m')})")

    # --- Build the merged left-join table (AIRS dates as the "left" table) ---
    left  = airs_monthly.rename("AIRS_TotCH4_A_mg_m2").to_frame()
    left.index.name = "date"
    right = cams_monthly.rename("CAMS_total_column_methane_mg_m2").to_frame()
    right.index.name = "date"
    merged = left.merge(right, how="left", left_index=True, right_index=True)

    merged.to_csv(MERGED_CSV, float_format="%.6g")
    print(f"\nWrote merged table: {MERGED_CSV}  (n = {len(merged)} months)")

    # --- Per-series trend analysis -------------------------------------------
    airs_dir = OUT_ROOT / "AIRS_TotCH4_A"
    cams_dir = OUT_ROOT / "CAMS_total_column_methane"

    print(f"\nProcessing AIRS3STD TotCH4_A ...")
    airs_res = run_single_series(
        label="AIRS3STD_TotCH4_A",
        pretty=r"AIRS3STD TotCH$_4$ (asc.) [mg m$^{-2}$]",
        color=COL_AIRS,
        monthly_raw=airs_monthly, coverage=airs_cov,
        out_dir=airs_dir)
    print(f"  -> {airs_dir}")

    print(f"\nProcessing CAMS total_column_methane ...")
    cams_res = run_single_series(
        label="CAMS_total_column_methane",
        pretty=r"CAMS Total Column CH$_4$ [mg m$^{-2}$]",
        color=COL_CAMS,
        monthly_raw=cams_monthly, coverage=cams_cov,
        out_dir=cams_dir)
    print(f"  -> {cams_dir}")

    # --- Combined z-score trend ---------------------------------------------
    z_path = OUT_ROOT / "05_zscore_trend_combined.png"
    plot_zscore_combined(
        airs_res["stl_trend_index"], airs_res["stl_trend_z"],
        "AIRS3STD TotCH$_4$ (asc.)",
        cams_res["stl_trend_index"], cams_res["stl_trend_z"],
        "CAMS Total Column CH$_4$",
        z_path)
    print(f"\nCombined z-score plot -> {z_path}")

    # --- Publication TIFF ---------------------------------------------------
    tiff_path = OUT_ROOT / "06_timeseries_scatter.tiff"
    plot_publication_tiff(merged, tiff_path)
    print(f"Publication TIFF      -> {tiff_path}")

    # --- One-line scoreboard CSV --------------------------------------------
    scoreboard = []
    for res in (airs_res, cams_res):
        row = {
            "series":      res["label"],
            "ref_value_mg_m2": res["ref_value"],
            "n_months":    res["n_months"],
            "n_imputed":   res["n_imputed"],
            "ar_p":        res["ar_p"],
            "ljungbox_p":  res["ljungbox_p"],
        }
        for r in res["df_three"].itertuples(index=False):
            row[f"slope_tau{r.tau:.2f}_mg_m2_per_yr"] = r.slope
            row[f"ci_low_tau{r.tau:.2f}_mg_m2_per_yr"] = r.ci_low
            row[f"ci_high_tau{r.tau:.2f}_mg_m2_per_yr"] = r.ci_high
        scoreboard.append(row)
    sb_path = OUT_ROOT / "_summary_both_series.csv"
    pd.DataFrame(scoreboard).to_csv(sb_path, index=False, float_format="%.6g")
    print(f"Scoreboard summary    -> {sb_path}")

    print("\nDone.")


if __name__ == "__main__":
    main()

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import pearsonr
from statsmodels.tsa.seasonal import STL
from docx import Document
from docx.shared import Mm, Pt
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

MERGED_CSV = Path(r"D:/PLOTS/CH4_comparison/ch4CAMSvsAIRSTD.csv")
OUT_DIR = Path(r"D:/PLOTS/CH4_comparison")
OUT_FIG = OUT_DIR / "08_timeseries_scatter_violin_trend_taylor_uncertaintyMC_final.tiff"
OUT_FIG_PNG = OUT_DIR / "08_timeseries_scatter_violin_trend_taylor_uncertaintyMC_final.png"
OUT_METRICS_CSV = OUT_DIR / "08_model_metrics_table_uncertaintyMC_final.csv"
OUT_METRICS_DOCX = OUT_DIR / "08_model_metrics_table_uncertaintyMC_final.docx"
OUT_TAYLOR_CSV = OUT_DIR / "08_taylor_statistics_uncertaintyMC_final.csv"
OUT_MC_CSV = OUT_DIR / "08_uncertainty_monte_carlo_summary_final.csv"
OUT_MC_DRAWS_CSV = OUT_DIR / "08_uncertainty_monte_carlo_draws_final.csv"

COL_AIRS = "#4E342E"
COL_CAMS = "#0277BD"
COL_MC = "#C62828"
UNIT_LABEL = r"mg m$^{-2}$"

STL_PERIOD = 12
STL_SEASONAL = 7
STL_TREND = 23
STL_ROBUST = True

AIRS_MONTHLY_REL_UNC = 0.0055
CAMS_MONTHLY_REL_UNC = 0.0135
N_MC = 20000
RNG_SEED = 42

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e6e6e6",
    "grid.linewidth": 0.6,
    "legend.frameon": False,
    "legend.fontsize": 7,
})

df = pd.read_csv(MERGED_CSV, parse_dates=["date"]).set_index("date")
df = df.rename(columns={
    "AIRS_TotCH4_A_mg_m2": "AIRS",
    "CAMS_total_column_methane_mg_m2": "CAMS"
})

pair = df[["AIRS", "CAMS"]].dropna().copy()
x = pair["AIRS"].to_numpy(float)
y = pair["CAMS"].to_numpy(float)

x_sigma = AIRS_MONTHLY_REL_UNC * x
y_sigma = CAMS_MONTHLY_REL_UNC * y
pair["AIRS_sigma"] = x_sigma
pair["CAMS_sigma"] = y_sigma

a, b = np.polyfit(x, y, 1)
r_all, p_all = pearsonr(x, y)
bias = y - x
ratio = y / x

rng_mc = np.random.default_rng(RNG_SEED)
mc = np.empty((N_MC, 9), dtype=float)

for i in range(N_MC):
    xsamp = rng_mc.normal(x, x_sigma)
    ysamp = rng_mc.normal(y, y_sigma)
    aa, bb = np.polyfit(xsamp, ysamp, 1)
    rr, pp = pearsonr(xsamp, ysamp)
    dd = ysamp - xsamp
    mc[i, 0] = aa
    mc[i, 1] = bb
    mc[i, 2] = rr
    mc[i, 3] = np.mean(dd)
    mc[i, 4] = np.median(dd)
    mc[i, 5] = np.sqrt(np.mean(dd ** 2))
    mc[i, 6] = np.mean(np.abs(dd))
    mc[i, 7] = np.median(ysamp / xsamp)
    mc[i, 8] = pp

mc_cols = [
    "slope",
    "intercept",
    "pearson_r",
    "mean_bias_mg_m2",
    "median_bias_mg_m2",
    "rmse_mg_m2",
    "mge_mg_m2",
    "median_ratio",
    "p_value"
]
mc_df = pd.DataFrame(mc, columns=mc_cols)
mc_df.to_csv(OUT_MC_DRAWS_CSV, index=False, float_format="%.8g")

def qstats(v):
    return pd.Series({
        "mean": np.nanmean(v),
        "median": np.nanmedian(v),
        "q025": np.nanpercentile(v, 2.5),
        "q975": np.nanpercentile(v, 97.5),
        "sd": np.nanstd(v, ddof=1)
    })

mc_summary = mc_df.apply(qstats).T.reset_index(names="quantity")
mc_summary.insert(0, "N_MC", N_MC)
mc_summary.insert(1, "AIRS_monthly_relative_uncertainty_percent", AIRS_MONTHLY_REL_UNC * 100)
mc_summary.insert(2, "CAMS_monthly_relative_uncertainty_percent", CAMS_MONTHLY_REL_UNC * 100)
mc_summary.to_csv(OUT_MC_CSV, index=False, float_format="%.8g")

vmin = np.nanmin([x.min(), y.min()])
vmax = np.nanmax([x.max(), y.max()])
pad = 0.06 * (vmax - vmin)
ylims = (vmin - pad, vmax + pad)

stl_a = STL(x, period=STL_PERIOD, seasonal=STL_SEASONAL, trend=STL_TREND, robust=STL_ROBUST).fit()
stl_c = STL(y, period=STL_PERIOD, seasonal=STL_SEASONAL, trend=STL_TREND, robust=STL_ROBUST).fit()

za = (stl_a.trend - np.nanmean(stl_a.trend)) / np.nanstd(stl_a.trend)
zc = (stl_c.trend - np.nanmean(stl_c.trend)) / np.nanstd(stl_c.trend)
r_trend, p_trend = pearsonr(za, zc)

def season_name(m):
    if m in (12, 1, 2):
        return "DJF"
    if m in (3, 4, 5):
        return "MAM"
    if m in (6, 7, 8):
        return "JJA"
    return "SON"

pair["season"] = [season_name(m) for m in pair.index.month]

def metrics(obs, mod, season, group):
    obs = np.asarray(obs, float)
    mod = np.asarray(mod, float)
    ok = np.isfinite(obs) & np.isfinite(mod)
    obs = obs[ok]
    mod = mod[ok]
    n = len(obs)
    d = mod - obs
    fac2 = np.mean((mod / obs >= 0.5) & (mod / obs <= 2.0)) if n else np.nan
    mb = np.mean(d) if n else np.nan
    mge = np.mean(np.abs(d)) if n else np.nan
    nmb = 100 * np.sum(d) / np.sum(obs) if n and np.sum(obs) != 0 else np.nan
    nmge = 100 * np.sum(np.abs(d)) / np.sum(obs) if n and np.sum(obs) != 0 else np.nan
    rmse = np.sqrt(np.mean(d ** 2)) if n else np.nan
    if n >= 3 and np.nanstd(obs) > 0 and np.nanstd(mod) > 0:
        rr, pp = pearsonr(obs, mod)
    else:
        rr, pp = np.nan, np.nan
    return {
        "season": season,
        "group": group,
        "n": n,
        "FAC2": fac2,
        "MB": mb,
        "MGE": mge,
        "NMB": nmb,
        "NMGE": nmge,
        "RMSE": rmse,
        "r": rr,
        "P-value": pp
    }

rows = [metrics(pair["AIRS"], pair["CAMS"], "All", "CAMS")]
for s in ["DJF", "MAM", "JJA", "SON"]:
    sub = pair[pair["season"] == s]
    rows.append(metrics(sub["AIRS"], sub["CAMS"], s, "CAMS"))

met = pd.DataFrame(rows)
met.to_csv(OUT_METRICS_CSV, index=False)

def taylor_stats(obs, mod, label):
    obs = np.asarray(obs, float)
    mod = np.asarray(mod, float)
    ok = np.isfinite(obs) & np.isfinite(mod)
    obs = obs[ok]
    mod = mod[ok]
    obs_c = obs - np.mean(obs)
    mod_c = mod - np.mean(mod)
    r = np.corrcoef(obs_c, mod_c)[0, 1]
    obs_sd = np.std(obs_c, ddof=1)
    mod_sd = np.std(mod_c, ddof=1)
    sdr = mod_sd / obs_sd
    crmsd = np.sqrt(np.mean((mod_c - obs_c) ** 2))
    crmsd_norm = crmsd / obs_sd
    theta = np.arccos(np.clip(r, 0, 1))
    return {
        "label": label,
        "r": r,
        "theta_rad": theta,
        "theta_deg": np.degrees(theta),
        "obs_sd": obs_sd,
        "mod_sd": mod_sd,
        "std_ratio": sdr,
        "centered_rmse": crmsd,
        "centered_rmse_norm": crmsd_norm
    }

taylor_records = [taylor_stats(pair["AIRS"], pair["CAMS"], "All")]
for s in ["DJF", "MAM", "JJA", "SON"]:
    sub = pair[pair["season"] == s]
    taylor_records.append(taylor_stats(sub["AIRS"], sub["CAMS"], s))

taylor_df = pd.DataFrame(taylor_records)
taylor_df.to_csv(OUT_TAYLOR_CSV, index=False)

r_ci = mc_summary.loc[mc_summary["quantity"] == "pearson_r", ["q025", "q975"]].to_numpy().ravel()
mb_ci = mc_summary.loc[mc_summary["quantity"] == "median_bias_mg_m2", ["q025", "q975"]].to_numpy().ravel()
ratio_ci = mc_summary.loc[mc_summary["quantity"] == "median_ratio", ["q025", "q975"]].to_numpy().ravel()

fig = plt.figure(figsize=(14.0, 9.05))
gs = fig.add_gridspec(
    2, 3,
    height_ratios=[1.0, 0.96],
    width_ratios=[1.55, 1.10, 0.86],
    hspace=0.66,
    wspace=0.38
)

ax = fig.add_subplot(gs[0, 0])
ax.plot(pair.index, pair["AIRS"], color=COL_AIRS, lw=1.35, label=r"AIRS3STD TotCH$_4$ (asc.)")
ax.plot(pair.index, pair["CAMS"], color=COL_CAMS, lw=1.35, label=r"CAMS total column CH$_4$")
ax.fill_between(pair.index, pair["AIRS"] - pair["AIRS_sigma"], pair["AIRS"] + pair["AIRS_sigma"], color=COL_AIRS, alpha=0.10, linewidth=0)
ax.fill_between(pair.index, pair["CAMS"] - pair["CAMS_sigma"], pair["CAMS"] + pair["CAMS_sigma"], color=COL_CAMS, alpha=0.10, linewidth=0)
ax.set_ylim(*ylims)
ax.set_ylabel(r"CH$_4$ total column (" + UNIT_LABEL + ")")
ax.set_xlabel("Date")
ax.set_title("(A) Monthly time series", loc="left", fontweight="bold")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.setp(ax.get_xticklabels(), rotation=90, ha="center")
ax.legend(loc="upper left")

ax = fig.add_subplot(gs[0, 1])
lo = np.nanmin([x.min(), y.min()])
hi = np.nanmax([x.max(), y.max()])
pad_sc = 0.04 * (hi - lo)
rng = np.array([lo - pad_sc, hi + pad_sc])
xs = np.linspace(rng[0], rng[1], 300)

ax.errorbar(
    x,
    y,
    xerr=x_sigma,
    yerr=y_sigma,
    fmt="none",
    ecolor="#b0b0b0",
    elinewidth=0.35,
    alpha=0.22,
    capsize=0,
    zorder=1
)
ax.scatter(
    x,
    y,
    s=12,
    color="#444444",
    alpha=0.55,
    edgecolor="none",
    label=f"Monthly pairs (n = {len(pair)})",
    zorder=3
)
ax.plot(rng, rng, ls="--", color="#888888", lw=1.0, label="1 : 1", zorder=2)

mc_pred = mc_df["slope"].to_numpy()[:, None] * xs[None, :] + mc_df["intercept"].to_numpy()[:, None]
mc_q = np.percentile(mc_pred, [2.5, 50, 97.5], axis=0)
ax.fill_between(xs, mc_q[0], mc_q[2], color=COL_MC, alpha=0.14, linewidth=0, label="MC regression 95% envelope", zorder=2)
ax.plot(xs, a * xs + b, color=COL_MC, lw=1.4, label="OLS fit", zorder=4)

ax.set_xlim(rng)
ax.set_ylim(rng)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel(r"AIRS3STD TotCH$_4$ (" + UNIT_LABEL + ")")
ax.set_ylabel(r"CAMS CH$_4$ (" + UNIT_LABEL + ")")
ax.set_title("(B) AIRS vs. CAMS with uncertainty propagation", loc="left", fontweight="bold")

stats_text = (
    f"y = {a:.2f}x + {b:.0f}\n"
    f"Pearson R = {r_all:.2f} [{r_ci[0]:.2f}, {r_ci[1]:.2f}]\n"
    f"Median bias = {np.nanmedian(bias):.0f} [{mb_ci[0]:.0f}, {mb_ci[1]:.0f}] {UNIT_LABEL}\n"
    f"Median ratio = {np.nanmedian(ratio):.3f} [{ratio_ci[0]:.3f}, {ratio_ci[1]:.3f}]"
)

ax.text(
    0.5,
    -0.31,
    stats_text,
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=7.6,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#999999", lw=0.8, alpha=0.96),
    clip_on=False,
    zorder=10
)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.52),
    ncol=2,
    fontsize=6.8,
    handlelength=1.6,
    columnspacing=0.9,
    borderaxespad=0.0
)

ax = fig.add_subplot(gs[0, 2])
parts = ax.violinplot([x, y], positions=[1, 2], widths=0.72, showmeans=False, showmedians=False, showextrema=False)

for body, col in zip(parts["bodies"], [COL_AIRS, COL_CAMS]):
    body.set_facecolor(col)
    body.set_edgecolor("black")
    body.set_alpha(0.65)
    body.set_linewidth(0.7)

for pos, vals in zip([1, 2], [x, y]):
    med = np.nanmedian(vals)
    q1 = np.nanpercentile(vals, 25)
    q3 = np.nanpercentile(vals, 75)
    ax.scatter(pos, med, s=30, color="white", edgecolor="black", linewidth=0.9, zorder=4)
    ax.vlines(pos, q1, q3, color="black", lw=2.1, zorder=3)
    ax.hlines([q1, q3], pos - 0.13, pos + 0.13, color="black", lw=1.2, zorder=3)
    ax.text(
        pos,
        q3 + 0.033 * (ylims[1] - ylims[0]),
        f"median {med:.0f}\nIQR {q1:.0f}–{q3:.0f}\n{UNIT_LABEL}",
        ha="center",
        va="bottom",
        fontsize=7.6
    )

ax.set_xticks([1, 2])
ax.set_xticklabels(["AIRS3STD", "CAMS"])
ax.set_ylim(*ylims)
ax.set_ylabel(r"CH$_4$ total column (" + UNIT_LABEL + ")")
ax.set_title("(C) Distribution", loc="left", fontweight="bold")
ax.grid(True, axis="y", color="#e6e6e6", lw=0.6)
ax.grid(False, axis="x")

ax = fig.add_subplot(gs[1, 0:2])
ax.plot(pair.index, za, color=COL_AIRS, lw=1.8, label=r"AIRS3STD TotCH$_4$ trend")
ax.plot(pair.index, zc, color=COL_CAMS, lw=1.8, label=r"CAMS CH$_4$ trend")
ax.axhline(0, ls="--", color="#888888", lw=0.8)
ax.set_ylabel("Standardised STL trend anomaly (z-score)")
ax.set_xlabel("Date")
ax.set_title("(D) Standardised STL trend", loc="left", fontweight="bold")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.setp(ax.get_xticklabels(), rotation=90, ha="center")
ax.text(
    0.02,
    0.95,
    f"Trend correlation R = {r_trend:.2f}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=8.5,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#999999", lw=0.8)
)
ax.legend(loc="lower right")

ax = fig.add_subplot(gs[1, 2], projection="polar")
ax.set_title("(E) Taylor diagram", loc="left", fontweight="bold", pad=16)
ax.set_theta_zero_location("E")
ax.set_theta_direction(1)
ax.set_thetamin(0)
ax.set_thetamax(90)

corr_ticks = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.99, 1.0])
theta_ticks = np.degrees(np.arccos(corr_ticks))
ax.set_thetagrids(theta_ticks, labels=[f"{c:.2g}" for c in corr_ticks])

rmax = max(1.6, np.nanmax(taylor_df["std_ratio"].to_numpy(float)) * 1.25, 1.05)
ax.set_rlim(0, rmax)
ax.set_rlabel_position(135)
ax.tick_params(axis="both", pad=5)

theta = np.linspace(0, np.pi / 2, 360)

for sd in [0.5, 1.0, 1.5]:
    if sd <= rmax:
        ax.plot(theta, np.full_like(theta, sd), color="#cfcfcf", lw=0.8, ls="--", zorder=1)

rs = np.linspace(0, rmax, 360)
ts = np.linspace(0, np.pi / 2, 360)
tt, rr = np.meshgrid(ts, rs)
crmsd = np.sqrt(1 + rr ** 2 - 2 * rr * np.cos(tt))
levels = [0.25, 0.50, 0.75, 1.00, 1.25, 1.50]
cs = ax.contour(tt, rr, crmsd, levels=levels, colors="#dddddd", linewidths=0.7, linestyles=":")
ax.clabel(cs, inline=True, fontsize=6, fmt="%.2g")

ax.scatter(0, 1, s=72, marker="*", color="black", label="AIRS reference", zorder=6)

markers = {"All": "o", "DJF": "s", "MAM": "^", "JJA": "D", "SON": "P"}

for _, row in taylor_df.iterrows():
    label = row["label"]
    theta_p = row["theta_rad"]
    radius_p = row["std_ratio"]
    ax.scatter(
        theta_p,
        radius_p,
        s=48,
        marker=markers[label],
        color=COL_CAMS,
        edgecolor="black",
        linewidth=0.6,
        label=f"{label}: r={row['r']:.2f}, σ={row['std_ratio']:.2f}",
        zorder=7
    )

ax.text(
    0.5,
    -0.24,
    "Correlation",
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=9,
    clip_on=False
)

ax.text(
    -0.25,
    0.5,
    "Normalised standard deviation",
    transform=ax.transAxes,
    ha="center",
    va="center",
    rotation=90,
    fontsize=9,
    clip_on=False
)

ax.legend(loc="upper right", bbox_to_anchor=(1.62, 1.17), fontsize=7.2)

fig.suptitle(
    r"CH$_4$ total column comparison over Mace Head with propagated monthly product uncertainty",
    y=0.985,
    fontsize=12,
    fontweight="bold"
)
fig.savefig(OUT_FIG, dpi=300, format="tiff", pil_kwargs={"compression": "tiff_lzw"})
fig.savefig(OUT_FIG_PNG, dpi=300)
plt.close(fig)

def set_cell_text(cell, text, bold=False, size=7):
    cell.text = ""
    p = cell.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = p.add_run(str(text))
    run.bold = bold
    run.font.size = Pt(size)
    cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER

def shade_cell(cell, fill):
    tcpr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)
    tcpr.append(shd)

def set_cell_width(cell, width_mm):
    tc = cell._tc
    tcpr = tc.get_or_add_tcPr()
    tcw = OxmlElement("w:tcW")
    tcw.set(qn("w:w"), str(int(width_mm * 56.7)))
    tcw.set(qn("w:type"), "dxa")
    tcpr.append(tcw)

doc = Document()
section = doc.sections[0]
section.orientation = WD_ORIENT.LANDSCAPE
section.page_width = Mm(297)
section.page_height = Mm(210)
section.top_margin = Mm(12)
section.bottom_margin = Mm(12)
section.left_margin = Mm(10)
section.right_margin = Mm(10)

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
run = p.add_run("Table X. Model performance metrics for CAMS CH₄ total column against AIRS3STD over Mace Head.")
run.bold = True
run.font.size = Pt(9)

cols = ["season", "group", "n", "FAC2", "MB", "MGE", "NMB", "NMGE", "RMSE", "r", "P-value"]
table = doc.add_table(rows=1, cols=len(cols))
table.alignment = WD_TABLE_ALIGNMENT.CENTER
table.style = "Table Grid"
table.autofit = False

widths = [15, 18, 11, 14, 18, 18, 16, 16, 18, 12, 18]

for j, c in enumerate(cols):
    set_cell_text(table.rows[0].cells[j], c, bold=True, size=7)
    shade_cell(table.rows[0].cells[j], "D9EAF7")
    set_cell_width(table.rows[0].cells[j], widths[j])

for _, row in met.iterrows():
    cells = table.add_row().cells
    values = [
        row["season"],
        row["group"],
        f"{int(row['n'])}",
        f"{row['FAC2']:.2f}",
        f"{row['MB']:.1f}",
        f"{row['MGE']:.1f}",
        f"{row['NMB']:.2f}",
        f"{row['NMGE']:.2f}",
        f"{row['RMSE']:.1f}",
        f"{row['r']:.2f}",
        f"{row['P-value']:.2e}"
    ]
    for j, val in enumerate(values):
        set_cell_text(cells[j], val, size=7)
        set_cell_width(cells[j], widths[j])

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.LEFT
run = p.add_run(
    "Units: MB, MGE and RMSE are in mg m⁻². NMB and NMGE are in %. "
    "FAC2 is the fraction of monthly CAMS values within a factor of two of AIRS3STD. "
    "Positive MB and NMB indicate higher CAMS columns. "
    f"Uncertainty propagation used independent Gaussian monthly relative uncertainties of {AIRS_MONTHLY_REL_UNC * 100:.2f}% for AIRS and {CAMS_MONTHLY_REL_UNC * 100:.2f}% for CAMS."
)
run.font.size = Pt(7)

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
run = p.add_run("Table Y. Monte Carlo uncertainty propagation for AIRS–CAMS monthly comparison.")
run.bold = True
run.font.size = Pt(9)

mc_cols_doc = ["quantity", "mean", "median", "q025", "q975", "sd"]
mc_table = doc.add_table(rows=1, cols=len(mc_cols_doc))
mc_table.alignment = WD_TABLE_ALIGNMENT.CENTER
mc_table.style = "Table Grid"
mc_table.autofit = False

mc_widths = [36, 22, 22, 22, 22, 22]

for j, c in enumerate(mc_cols_doc):
    set_cell_text(mc_table.rows[0].cells[j], c, bold=True, size=7)
    shade_cell(mc_table.rows[0].cells[j], "D9EAF7")
    set_cell_width(mc_table.rows[0].cells[j], mc_widths[j])

for _, row in mc_summary.iterrows():
    cells = mc_table.add_row().cells
    vals = [
        row["quantity"],
        f"{row['mean']:.4g}",
        f"{row['median']:.4g}",
        f"{row['q025']:.4g}",
        f"{row['q975']:.4g}",
        f"{row['sd']:.4g}"
    ]
    for j, val in enumerate(vals):
        set_cell_text(cells[j], val, size=7)
        set_cell_width(cells[j], mc_widths[j])

doc.save(OUT_METRICS_DOCX)

print(f"Saved figure TIFF: {OUT_FIG}")
print(f"Saved figure PNG: {OUT_FIG_PNG}")
print(f"Saved metrics CSV: {OUT_METRICS_CSV}")
print(f"Saved Word table: {OUT_METRICS_DOCX}")
print(f"Saved Taylor statistics CSV: {OUT_TAYLOR_CSV}")
print(f"Saved Monte Carlo summary CSV: {OUT_MC_CSV}")
print(f"Saved Monte Carlo draws CSV: {OUT_MC_DRAWS_CSV}")
print("\nCentral-value metrics:")
print(met.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nTaylor statistics:")
print(taylor_df.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nMonte Carlo uncertainty propagation:")
print(mc_summary.to_string(index=False, float_format=lambda v: f"{v:.4g}"))

Saved figure TIFF: D:\PLOTS\CH4_comparison\08_timeseries_scatter_violin_trend_taylor_uncertaintyMC_final.tiff
Saved figure PNG: D:\PLOTS\CH4_comparison\08_timeseries_scatter_violin_trend_taylor_uncertaintyMC_final.png
Saved metrics CSV: D:\PLOTS\CH4_comparison\08_model_metrics_table_uncertaintyMC_final.csv
Saved Word table: D:\PLOTS\CH4_comparison\08_model_metrics_table_uncertaintyMC_final.docx
Saved Taylor statistics CSV: D:\PLOTS\CH4_comparison\08_taylor_statistics_uncertaintyMC_final.csv
Saved Monte Carlo summary CSV: D:\PLOTS\CH4_comparison\08_uncertainty_monte_carlo_summary_final.csv
Saved Monte Carlo draws CSV: D:\PLOTS\CH4_comparison\08_uncertainty_monte_carlo_draws_final.csv

Central-value metrics:
season group   n  FAC2  MB  MGE  NMB  NMGE  RMSE     r  P-value
   All  CAMS 272     1 375  376  3.9   3.9   400  0.59 6.62e-27
   DJF  CAMS  68     1 524  524 5.51  5.51   533 0.729 1.72e-12
   MAM  CAMS  69     1 367  367 3.82  3.82   379 0.808 5.01e-17
   JJA  CAMS  69     1 268  

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import pearsonr
from docx import Document
from docx.shared import Mm, Pt
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

MERGED_CSV = Path(r"D:/PLOTS/CH4_comparison/ch4CAMSvsAIRSTD.csv")
OUT_DIR = Path(r"D:/PLOTS/CH4_comparison/RAT_vs_quantilemapping")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FIG_TIFF = OUT_DIR / "RAT_vs_QQ_bias_correction_comparison.tiff"
OUT_FIG_PNG = OUT_DIR / "RAT_vs_QQ_bias_correction_comparison.png"
OUT_DATA = OUT_DIR / "RAT_vs_QQ_corrected_monthly_data.csv"
OUT_TABLE_CSV = OUT_DIR / "RAT_vs_QQ_summary_metrics.csv"
OUT_TABLE_DOCX = OUT_DIR / "RAT_vs_QQ_summary_metrics.docx"

COL_AIRS = "#4E342E"
COL_RAW = "#0277BD"
COL_QQ = "#00897B"
COL_RAT_ADD = "#7B1FA2"
COL_RAT_MULTI = "#C62828"
UNIT_LABEL = r"mg m$^{-2}$"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e6e6e6",
    "grid.linewidth": 0.6,
    "legend.frameon": False,
    "legend.fontsize": 8,
})

df = pd.read_csv(MERGED_CSV, parse_dates=["date"]).set_index("date")
df = df.rename(columns={
    "AIRS_TotCH4_A_mg_m2": "AIRS",
    "CAMS_total_column_methane_mg_m2": "CAMS"
})
df = df[["AIRS", "CAMS"]].dropna().copy()

def season_name(m):
    if m in (12, 1, 2):
        return "DJF"
    if m in (3, 4, 5):
        return "MAM"
    if m in (6, 7, 8):
        return "JJA"
    return "SON"

df["season"] = [season_name(m) for m in df.index.month]

def qq_mapping(obs, mod):
    obs = np.asarray(obs, float)
    mod = np.asarray(mod, float)
    o = np.sort(obs[np.isfinite(obs)])
    m = np.sort(mod[np.isfinite(mod)])
    q_m = np.linspace(0, 1, len(m))
    q_o = np.linspace(0, 1, len(o))
    p = np.interp(mod, m, q_m, left=0, right=1)
    return np.interp(p, q_o, o, left=o[0], right=o[-1])

def rat_add(obs, mod):
    return mod + (np.nanmean(obs) - np.nanmean(mod))

def rat_multi(obs, mod):
    return mod * (np.nansum(obs) / np.nansum(mod))

df["CAMS_QQ"] = qq_mapping(df["AIRS"], df["CAMS"])
df["CAMS_RAT_add"] = rat_add(df["AIRS"], df["CAMS"])
df["CAMS_RAT_multi"] = rat_multi(df["AIRS"], df["CAMS"])

df["bias_raw"] = df["CAMS"] - df["AIRS"]
df["bias_QQ"] = df["CAMS_QQ"] - df["AIRS"]
df["bias_RAT_add"] = df["CAMS_RAT_add"] - df["AIRS"]
df["bias_RAT_multi"] = df["CAMS_RAT_multi"] - df["AIRS"]

df["ratio_raw"] = df["CAMS"] / df["AIRS"]
df["ratio_QQ"] = df["CAMS_QQ"] / df["AIRS"]
df["ratio_RAT_add"] = df["CAMS_RAT_add"] / df["AIRS"]
df["ratio_RAT_multi"] = df["CAMS_RAT_multi"] / df["AIRS"]

df.to_csv(OUT_DATA, float_format="%.6f")

def metrics(obs, mod, method):
    obs = np.asarray(obs, float)
    mod = np.asarray(mod, float)
    ok = np.isfinite(obs) & np.isfinite(mod)
    obs = obs[ok]
    mod = mod[ok]
    n = len(obs)
    d = mod - obs
    rr, pp = pearsonr(obs, mod) if n >= 3 and np.nanstd(obs) > 0 and np.nanstd(mod) > 0 else (np.nan, np.nan)
    return {
        "method": method,
        "n": n,
        "median_obs_mg_m2": np.nanmedian(obs),
        "median_model_mg_m2": np.nanmedian(mod),
        "FAC2": np.mean((mod / obs >= 0.5) & (mod / obs <= 2.0)),
        "MB_mg_m2": np.nanmean(d),
        "median_bias_mg_m2": np.nanmedian(d),
        "MGE_mg_m2": np.nanmean(np.abs(d)),
        "NMB_percent": 100 * np.nansum(d) / np.nansum(obs),
        "NMGE_percent": 100 * np.nansum(np.abs(d)) / np.nansum(obs),
        "RMSE_mg_m2": np.sqrt(np.nanmean(d ** 2)),
        "r": rr,
        "P-value": pp,
        "median_ratio": np.nanmedian(mod / obs),
        "IQR_bias_low_mg_m2": np.nanpercentile(d, 25),
        "IQR_bias_high_mg_m2": np.nanpercentile(d, 75)
    }

summary = pd.DataFrame([
    metrics(df["AIRS"], df["CAMS"], "Raw CAMS"),
    metrics(df["AIRS"], df["CAMS_QQ"], "QQ mapping"),
    metrics(df["AIRS"], df["CAMS_RAT_add"], "RAT-add"),
    metrics(df["AIRS"], df["CAMS_RAT_multi"], "RAT-multi")
])

summary.to_csv(OUT_TABLE_CSV, index=False, float_format="%.6g")

methods = ["Raw CAMS", "QQ mapping", "RAT-add", "RAT-multi"]
series_cols = ["CAMS", "CAMS_QQ", "CAMS_RAT_add", "CAMS_RAT_multi"]
bias_cols = ["bias_raw", "bias_QQ", "bias_RAT_add", "bias_RAT_multi"]
colors = [COL_RAW, COL_QQ, COL_RAT_ADD, COL_RAT_MULTI]

obs = df["AIRS"].to_numpy(float)
raw = df["CAMS"].to_numpy(float)

vmin = np.nanmin(df[["AIRS"] + series_cols].to_numpy())
vmax = np.nanmax(df[["AIRS"] + series_cols].to_numpy())
pad = 0.06 * (vmax - vmin)
ylims = (vmin - pad, vmax + pad)

bmin = np.nanmin(df[bias_cols].to_numpy())
bmax = np.nanmax(df[bias_cols].to_numpy())
bpad = 0.10 * (bmax - bmin)
bylims = (bmin - bpad, bmax + bpad)

fig = plt.figure(figsize=(14.2, 9.2))
gs = fig.add_gridspec(3, 3, height_ratios=[1.05, 0.95, 0.95], width_ratios=[1.35, 1.0, 1.0], hspace=0.47, wspace=0.32)

ax = fig.add_subplot(gs[0, 0])
ax.plot(df.index, df["AIRS"], color=COL_AIRS, lw=1.35, label=r"AIRS3STD TotCH$_4$")
for col, lab, c in zip(series_cols, methods, colors):
    ax.plot(df.index, df[col], color=c, lw=1.05 if lab == "Raw CAMS" else 1.25, alpha=0.70 if lab == "Raw CAMS" else 0.95, label=lab)
ax.set_ylim(*ylims)
ax.set_title("(A) Monthly CH$_4$ columns", loc="left", fontweight="bold")
ax.set_ylabel(r"CH$_4$ total column (" + UNIT_LABEL + ")")
ax.set_xlabel("Date")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.setp(ax.get_xticklabels(), rotation=90, ha="center")
ax.legend(loc="upper left", ncol=1)

ax = fig.add_subplot(gs[0, 1])
lo = np.nanmin(df[["AIRS"] + series_cols].to_numpy())
hi = np.nanmax(df[["AIRS"] + series_cols].to_numpy())
spad = 0.04 * (hi - lo)
rng = np.array([lo - spad, hi + spad])
xs = np.linspace(rng[0], rng[1], 300)
ax.plot(rng, rng, ls="--", color="#888888", lw=1.0, label="1 : 1")
for col, lab, c in zip(series_cols, methods, colors):
    yy = df[col].to_numpy(float)
    rr, _ = pearsonr(obs, yy)
    aa, bb = np.polyfit(obs, yy, 1)
    ax.scatter(obs, yy, s=10, color=c, alpha=0.28 if lab == "Raw CAMS" else 0.45, edgecolor="none", label=f"{lab}, R={rr:.2f}")
    ax.plot(xs, aa * xs + bb, color=c, lw=1.1)
ax.set_xlim(rng)
ax.set_ylim(rng)
ax.set_aspect("equal", adjustable="box")
ax.set_title("(B) AIRS vs. raw/corrected CAMS", loc="left", fontweight="bold")
ax.set_xlabel(r"AIRS3STD TotCH$_4$ (" + UNIT_LABEL + ")")
ax.set_ylabel(r"CAMS CH$_4$ (" + UNIT_LABEL + ")")
ax.legend(loc="lower right", fontsize=7)

ax = fig.add_subplot(gs[0, 2])
vals = [df[c].to_numpy(float) for c in bias_cols]
parts = ax.violinplot(vals, positions=np.arange(1, 5), widths=0.74, showmeans=False, showmedians=False, showextrema=False)
for body, c in zip(parts["bodies"], colors):
    body.set_facecolor(c)
    body.set_edgecolor("black")
    body.set_alpha(0.62)
    body.set_linewidth(0.7)
for pos, v in zip(np.arange(1, 5), vals):
    med = np.nanmedian(v)
    q1 = np.nanpercentile(v, 25)
    q3 = np.nanpercentile(v, 75)
    ax.scatter(pos, med, s=28, color="white", edgecolor="black", linewidth=0.9, zorder=4)
    ax.vlines(pos, q1, q3, color="black", lw=2.0, zorder=3)
    ax.hlines([q1, q3], pos - 0.13, pos + 0.13, color="black", lw=1.1, zorder=3)
ax.axhline(0, color="#777777", ls="--", lw=0.8)
ax.set_xticks(np.arange(1, 5))
ax.set_xticklabels(methods, rotation=25, ha="right")
ax.set_ylim(*bylims)
ax.set_title("(C) Bias distributions", loc="left", fontweight="bold")
ax.set_ylabel(r"Bias vs. AIRS (" + UNIT_LABEL + ")")

ax = fig.add_subplot(gs[1, 0:2])
for bc, lab, c in zip(bias_cols, methods, colors):
    ax.plot(df.index, df[bc], color=c, lw=1.05 if lab == "Raw CAMS" else 1.20, alpha=0.70 if lab == "Raw CAMS" else 0.95, label=lab)
ax.axhline(0, color="#777777", ls="--", lw=0.8)
ax.set_ylim(*bylims)
ax.set_title("(D) Monthly bias", loc="left", fontweight="bold")
ax.set_ylabel(r"Bias vs. AIRS (" + UNIT_LABEL + ")")
ax.set_xlabel("Date")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.setp(ax.get_xticklabels(), rotation=90, ha="center")
ax.legend(loc="upper left", ncol=2)

ax = fig.add_subplot(gs[1, 2])
xpos = np.arange(len(methods))
rmse = summary["RMSE_mg_m2"].to_numpy(float)
mb = summary["MB_mg_m2"].to_numpy(float)
mge = summary["MGE_mg_m2"].to_numpy(float)
w = 0.24
ax.bar(xpos - w, rmse, width=w, color="#666666", alpha=0.75, label="RMSE")
ax.bar(xpos, mge, width=w, color="#999999", alpha=0.75, label="MGE")
ax.bar(xpos + w, np.abs(mb), width=w, color="#C62828", alpha=0.70, label="|MB|")
ax.set_xticks(xpos)
ax.set_xticklabels(methods, rotation=25, ha="right")
ax.set_ylabel(r"Error magnitude (" + UNIT_LABEL + ")")
ax.set_title("(E) Error metrics", loc="left", fontweight="bold")
ax.legend(loc="upper right")
for i, v in enumerate(rmse):
    ax.text(i - w, v, f"{v:.1f}", ha="center", va="bottom", fontsize=7)

ax = fig.add_subplot(gs[2, 0])
order = ["DJF", "MAM", "JJA", "SON"]
base = np.arange(len(order)) * 1.25
offsets = [-0.36, -0.12, 0.12, 0.36]
season_medians = []
for bc in bias_cols:
    season_medians.append([np.nanmedian(df.loc[df["season"] == s, bc]) for s in order])
season_medians = np.asarray(season_medians)
for i, lab, c in zip(range(4), methods, colors):
    ax.bar(base + offsets[i], season_medians[i], width=0.22, color=c, alpha=0.75, label=lab)
ax.axhline(0, color="#777777", ls="--", lw=0.8)
ax.set_xticks(base)
ax.set_xticklabels(order)
ax.set_title("(F) Seasonal median bias", loc="left", fontweight="bold")
ax.set_ylabel(r"Median bias (" + UNIT_LABEL + ")")
ax.legend(loc="best", fontsize=7)

ax = fig.add_subplot(gs[2, 1])
for col, lab, c in zip(series_cols, methods, colors):
    q_obs = np.quantile(df["AIRS"], np.linspace(0.01, 0.99, 99))
    q_mod = np.quantile(df[col], np.linspace(0.01, 0.99, 99))
    ax.plot(q_obs, q_mod, color=c, lw=1.4, label=lab)
ax.plot(rng, rng, ls="--", color="#888888", lw=1.0)
ax.set_xlim(rng)
ax.set_ylim(rng)
ax.set_aspect("equal", adjustable="box")
ax.set_title("(G) Quantile-quantile comparison", loc="left", fontweight="bold")
ax.set_xlabel(r"AIRS quantiles (" + UNIT_LABEL + ")")
ax.set_ylabel(r"CAMS quantiles (" + UNIT_LABEL + ")")
ax.legend(loc="lower right", fontsize=7)

ax = fig.add_subplot(gs[2, 2])
nmb = summary["NMB_percent"].to_numpy(float)
nmge = summary["NMGE_percent"].to_numpy(float)
xpos = np.arange(len(methods))
ax.bar(xpos - 0.16, nmb, width=0.32, color="#1565C0", alpha=0.72, label="NMB")
ax.bar(xpos + 0.16, nmge, width=0.32, color="#EF6C00", alpha=0.72, label="NMGE")
ax.axhline(0, color="#777777", ls="--", lw=0.8)
ax.set_xticks(xpos)
ax.set_xticklabels(methods, rotation=25, ha="right")
ax.set_ylabel("Normalised error (%)")
ax.set_title("(H) Normalised bias and gross error", loc="left", fontweight="bold")
ax.legend(loc="upper right")

fig.suptitle(r"Comparison of QQ mapping and RAT bias correction for CAMS CH$_4$ total columns", y=0.992, fontsize=12, fontweight="bold")
fig.savefig(OUT_FIG_TIFF, dpi=300, format="tiff", pil_kwargs={"compression": "tiff_lzw"})
fig.savefig(OUT_FIG_PNG, dpi=300)
plt.close(fig)

def set_cell_text(cell, text, bold=False, size=7):
    cell.text = ""
    p = cell.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = p.add_run(str(text))
    run.bold = bold
    run.font.size = Pt(size)
    cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER

def shade_cell(cell, fill):
    tcpr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)
    tcpr.append(shd)

def set_cell_width(cell, width_mm):
    tc = cell._tc
    tcpr = tc.get_or_add_tcPr()
    tcw = OxmlElement("w:tcW")
    tcw.set(qn("w:w"), str(int(width_mm * 56.7)))
    tcw.set(qn("w:type"), "dxa")
    tcpr.append(tcw)

doc = Document()
section = doc.sections[0]
section.orientation = WD_ORIENT.LANDSCAPE
section.page_width = Mm(297)
section.page_height = Mm(210)
section.top_margin = Mm(11)
section.bottom_margin = Mm(11)
section.left_margin = Mm(9)
section.right_margin = Mm(9)

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
run = p.add_run("Table X. Summary metrics comparing raw CAMS, QQ mapping and RAT bias correction against AIRS3STD CH₄ total column over Mace Head.")
run.bold = True
run.font.size = Pt(9)

cols = [
    "method", "n", "FAC2", "MB", "Med. bias", "MGE", "NMB", "NMGE",
    "RMSE", "r", "P-value", "Med. ratio"
]
table = doc.add_table(rows=1, cols=len(cols))
table.alignment = WD_TABLE_ALIGNMENT.CENTER
table.style = "Table Grid"
table.autofit = False
widths = [26, 10, 13, 16, 18, 16, 15, 15, 16, 11, 18, 17]

for j, c in enumerate(cols):
    set_cell_text(table.rows[0].cells[j], c, bold=True, size=7)
    shade_cell(table.rows[0].cells[j], "D9EAF7")
    set_cell_width(table.rows[0].cells[j], widths[j])

for _, row in summary.iterrows():
    cells = table.add_row().cells
    vals = [
        row["method"],
        f"{int(row['n'])}",
        f"{row['FAC2']:.2f}",
        f"{row['MB_mg_m2']:.1f}",
        f"{row['median_bias_mg_m2']:.1f}",
        f"{row['MGE_mg_m2']:.1f}",
        f"{row['NMB_percent']:.2f}",
        f"{row['NMGE_percent']:.2f}",
        f"{row['RMSE_mg_m2']:.1f}",
        f"{row['r']:.2f}",
        f"{row['P-value']:.2e}",
        f"{row['median_ratio']:.3f}"
    ]
    for j, val in enumerate(vals):
        set_cell_text(cells[j], val, size=7)
        set_cell_width(cells[j], widths[j])

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.LEFT
run = p.add_run(
    "Reference: AIRS3STD TotCH₄. Model: CAMS raw or corrected CAMS. "
    "Units: MB, median bias, MGE and RMSE are mg m⁻². NMB and NMGE are %. "
    "FAC2 is the fraction of monthly CAMS values within a factor of two of AIRS3STD. "
    "Positive MB and NMB indicate higher CAMS columns."
)
run.font.size = Pt(7)

doc.save(OUT_TABLE_DOCX)

print(f"Saved folder: {OUT_DIR}")
print(f"Saved plot TIFF: {OUT_FIG_TIFF}")
print(f"Saved plot PNG: {OUT_FIG_PNG}")
print(f"Saved corrected data: {OUT_DATA}")
print(f"Saved summary CSV: {OUT_TABLE_CSV}")
print(f"Saved Word table: {OUT_TABLE_DOCX}")
print(summary.to_string(index=False, float_format=lambda v: f"{v:.4g}"))